# Notebook 12 – Feature Leakage

## Topic 1 – What is Feature Leakage?


### Step 1 – Understand the Concept

Feature Leakage occurs when a Machine Learning model receives information during training that would not legitimately be available when making predictions in the real world.

This can make the model appear more accurate than it actually is.

The model may learn information about the target from data that should not have been used as input features.

Common causes of feature leakage:

1. Using information from the target variable.
2. Using information recorded after the outcome.
3. Using future information to predict the past.
4. Calculating features using the complete dataset before splitting.
5. Using target information incorrectly during feature encoding.

Feature leakage can produce unrealistic model performance and poor predictions on new data.


### Step 2 – Demonstrate the Concept

Real-world example: Loan Default Prediction

A bank wants to predict whether a customer will default on a loan.

Target:
- Default (Yes or No)

Available input features at application time:
- Customer Age
- Monthly Income
- Credit Score
- Loan Amount

Incorrect feature:
- Recovery Amount

Recovery Amount is recorded after a customer defaults.

Using Recovery Amount to predict whether the customer will default creates leakage because this information is not available at prediction time.

Correct approach:

Use only information available when the loan application is evaluated.


### Step 3 – Implement the Concept

We will create a sample dataset containing a leaked feature called Recovery Amount.

We will compare a model trained with the leaked feature against a model trained using only legitimate application-time features.

This is a controlled demonstration of target leakage.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create sample loan dataset
np.random.seed(42)

n = 1000

income = np.random.randint(20000, 100000, n)
credit_score = np.random.randint(450, 850, n)
loan_amount = np.random.randint(10000, 500000, n)

# Create default target
default = (
    (credit_score < 600) |
    (loan_amount > income * 4)
).astype(int)

# This feature is recorded after the outcome
recovery_amount = np.where(
    default == 1,
    np.random.randint(10000, 200000, n),
    0
)

loan_df = pd.DataFrame({
    "Income": income,
    "CreditScore": credit_score,
    "LoanAmount": loan_amount,
    "RecoveryAmount": recovery_amount,
    "Default": default
})

# Separate leaked and legitimate features
X_leaked = loan_df.drop("Default", axis=1)

X_correct = loan_df[
    ["Income", "CreditScore", "LoanAmount"]
]

y = loan_df["Default"]

# Split data
X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_leaked,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_good = X_train_bad[
    ["Income", "CreditScore", "LoanAmount"]
]

X_test_good = X_test_bad[
    ["Income", "CreditScore", "LoanAmount"]
]

# Train model with leaked feature
bad_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

bad_model.fit(X_train_bad, y_train)

bad_predictions = bad_model.predict(X_test_bad)

# Train model without leaked feature
good_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

good_model.fit(X_train_good, y_train)

good_predictions = good_model.predict(X_test_good)

# Compare performance
print("Accuracy with Leaked Feature:",
      accuracy_score(y_test, bad_predictions))

print("Accuracy without Leaked Feature:",
      accuracy_score(y_test, good_predictions))

Accuracy with Leaked Feature: 1.0
Accuracy without Leaked Feature: 0.975


### Output Explanation

The first model uses RecoveryAmount, which is information recorded after the default outcome.

The second model uses only features available when evaluating the loan application.

The model with the leaked feature may achieve unrealistically high accuracy.

Important:

Both models use the same random train-test split. Therefore, this experiment demonstrates target leakage, not train-test contamination.

The leaked feature is not valid for real-world prediction, even if the model achieves high accuracy.

## Topic 2 – Target Leakage


### Step 1 – Understand the Concept

Target Leakage happens when an input feature contains information that directly or indirectly reveals the target variable.

The model learns from information that would not legitimately be available at prediction time.

Examples:

- Using final exam results to predict whether a student will pass.
- Using claim settlement amount to predict whether an insurance claim will be approved.
- Using recovery amount to predict loan default.
- Using a target-derived column as an input feature.

Target leakage can make model performance look extremely high.

However, the model may fail when used on new real-world records.


### Step 2 – Demonstrate the Concept

Real-world example: Insurance Claim Prediction

An insurance company wants to predict whether a claim will be approved.

Target:
- Claim Approved (Yes or No)

Available features when the claim is submitted:
- Customer Age
- Policy Type
- Claim Amount
- Policy Duration

Incorrect feature:
- Final Settlement Status

Final Settlement Status is created after the claim review process.

Using this information to predict approval reveals information about the outcome.

Correct approach:

Use only features available when the claim is submitted or at the intended prediction time.


### Step 3 – Implement the Concept

We will create a dataset containing a target-derived feature called ApprovalFlag.

We will compare incorrect and correct feature sets.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create sample insurance dataset
np.random.seed(42)

n = 500

age = np.random.randint(20, 70, n)
claim_amount = np.random.randint(5000, 100000, n)
policy_duration = np.random.randint(1, 15, n)

# Create target
claim_approved = (
    (claim_amount < 60000) &
    (policy_duration >= 3)
).astype(int)

# Target-derived feature: leakage
approval_flag = claim_approved.copy()

insurance_df = pd.DataFrame({
    "Age": age,
    "ClaimAmount": claim_amount,
    "PolicyDuration": policy_duration,
    "ApprovalFlag": approval_flag,
    "ClaimApproved": claim_approved
})

# Incorrect features
X_bad = insurance_df.drop("ClaimApproved", axis=1)

# Correct features
X_good = insurance_df[
    ["Age", "ClaimAmount", "PolicyDuration"]
]

y = insurance_df["ClaimApproved"]

# Split dataset
X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_bad,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_good = X_train_bad[X_good.columns]
X_test_good = X_test_bad[X_good.columns]

# Incorrect model
bad_model = RandomForestClassifier(random_state=42)

bad_model.fit(X_train_bad, y_train)

bad_pred = bad_model.predict(X_test_bad)

# Correct model
good_model = RandomForestClassifier(random_state=42)

good_model.fit(X_train_good, y_train)

good_pred = good_model.predict(X_test_good)

print("Accuracy with Target Leakage:",
      accuracy_score(y_test, bad_pred))

print("Accuracy without Target Leakage:",
      accuracy_score(y_test, good_pred))

Accuracy with Target Leakage: 1.0
Accuracy without Target Leakage: 1.0


### Output Explanation

The leaked feature ApprovalFlag directly contains the target information.

Therefore, the incorrect model can predict the target using information that would not be available at prediction time.

The correct model uses only the three legitimate input features.

The correct model may have lower accuracy, but its evaluation is more meaningful for real-world prediction.

## Topic 3 – Train-Test Leakage

### Step 1 – Understand the Concept

Train-Test Leakage occurs when information from the test dataset influences the training process.

The test dataset must remain unseen during model training and preprocessing.

Common examples:

1. Applying StandardScaler to the complete dataset before splitting.
2. Selecting features using the complete dataset before splitting.
3. Filling missing values using statistics calculated from the complete dataset.
4. Performing feature selection using test labels.

Why is this a problem?

The model indirectly receives information from the test dataset.

As a result, test performance may not represent how the model performs on genuinely unseen data.

Correct approach:

- Split the dataset first.
- Fit preprocessing steps using training data only.
- Apply the fitted transformations to test data.


### Step 2 – Demonstrate the Concept

Real-world example: Customer Churn Prediction

A telecom company wants to predict customer churn.

It has 10,000 customer records.

Incorrect approach:

1. Calculate the average monthly charges using all 10,000 records.
2. Fill missing values using that average.
3. Split the dataset into training and testing data.

The training process has already used information from the test dataset.

Correct approach:

1. Split the dataset.
2. Calculate the average monthly charges using training data only.
3. Fill missing training values using the training average.
4. Fill missing testing values using the same training average.

This keeps the test dataset independent.



### Step 3 – Implement the Concept

We will compare two approaches for filling missing values.

Incorrect:
Calculate the mean before splitting the data.

Correct:
Split the data first and calculate the mean using only training data.

In [3]:
# Create sample customer dataset
customer_data = pd.DataFrame({
    "Age": [22, 25, 30, 35, 40, 45, 50, 55, 60, 65],
    "MonthlyCharges": [
        1000, 1200, np.nan, 1500, 1800,
        2000, np.nan, 2400, 2600, 10000
    ],
    "Churn": [0, 0, 0, 1, 0, 1, 1, 1, 0, 1]
})

# Incorrect approach: calculate mean before splitting
bad_df = customer_data.copy()

bad_mean = bad_df["MonthlyCharges"].mean()

bad_df["MonthlyCharges"] = bad_df[
    "MonthlyCharges"
].fillna(bad_mean)

X_bad = bad_df.drop("Churn", axis=1)
y = bad_df["Churn"]

X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_bad,
    y,
    test_size=0.3,
    random_state=42
)

# Correct approach: split before calculating the mean
X = customer_data.drop("Churn", axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

train_mean = X_train["MonthlyCharges"].mean()

X_train_correct = X_train.copy()
X_test_correct = X_test.copy()

X_train_correct["MonthlyCharges"] = (
    X_train_correct["MonthlyCharges"].fillna(train_mean)
)

X_test_correct["MonthlyCharges"] = (
    X_test_correct["MonthlyCharges"].fillna(train_mean)
)

print("Incorrect Mean:", round(bad_mean, 2))

print("Correct Training Mean:", round(train_mean, 2))

print("\nCorrected Training Data:")
print(X_train_correct)

print("\nCorrected Testing Data:")
print(X_test_correct)

Incorrect Mean: 2812.5
Correct Training Mean: 3340.0

Corrected Training Data:
   Age  MonthlyCharges
0   22          1000.0
7   55          2400.0
2   30          3340.0
9   65         10000.0
4   40          1800.0
3   35          1500.0
6   50          3340.0

Corrected Testing Data:
   Age  MonthlyCharges
8   60          2600.0
1   25          1200.0
5   45          2000.0


### Output Explanation

The incorrect approach calculates the mean before splitting the dataset.

Therefore, the mean can include information from records that later become part of the test dataset.

The correct approach calculates the mean only from training data.

The same training mean is then used to fill missing values in both training and testing data.

Important:

This example demonstrates leakage through preprocessing. It does not mean every preprocessing operation before splitting always causes leakage. The important rule is that learned preprocessing statistics must not use test data.

## Topic 4 – Temporal Leakage


### Step 1 – Understand the Concept

Temporal Leakage occurs when a model uses information from the future to predict an event in the past or at an earlier prediction time.

It is common in time-dependent datasets.

Examples:

- Using tomorrow's sales to predict today's sales.
- Using next month's customer activity to predict current churn.
- Using future stock prices to predict today's price.
- Using information recorded after a transaction to predict whether that transaction will succeed.

Correct approach:

Features must contain only information available at the prediction timestamp.

For time-dependent problems, training and testing should generally respect chronological order.



### Step 2 – Demonstrate the Concept

Real-world example: Sales Forecasting

A retail company wants to predict sales for the next day.

Available information at the end of today:
- Today's Sales
- Previous Day Sales
- Previous Week Sales

Incorrect feature:
- Actual Sales of Tomorrow

Tomorrow's actual sales are not available when the prediction is made.

Using tomorrow's sales as an input feature creates temporal leakage.

Correct approach:

Use only past sales information to predict future sales.



### Step 3 – Implement the Concept

We will create a daily sales dataset.

We will demonstrate:

Incorrect:
Using future sales as an input feature.

Correct:
Using previous day's sales as an input feature.

In [4]:
# Create daily sales data
sales_df = pd.DataFrame({
    "Date": pd.date_range(
        start="2026-01-01",
        periods=10,
        freq="D"
    ),
    "Sales": [
        100, 120, 130, 110, 150,
        160, 170, 180, 200, 210
    ]
})

# Incorrect feature: next day's actual sales
sales_df["FutureSales"] = sales_df["Sales"].shift(-1)

# Correct feature: previous day's sales
sales_df["PreviousDaySales"] = sales_df["Sales"].shift(1)

print(sales_df)

        Date  Sales  FutureSales  PreviousDaySales
0 2026-01-01    100        120.0               NaN
1 2026-01-02    120        130.0             100.0
2 2026-01-03    130        110.0             120.0
3 2026-01-04    110        150.0             130.0
4 2026-01-05    150        160.0             110.0
5 2026-01-06    160        170.0             150.0
6 2026-01-07    170        180.0             160.0
7 2026-01-08    180        200.0             170.0
8 2026-01-09    200        210.0             180.0
9 2026-01-10    210          NaN             200.0


### Output Explanation

The FutureSales column contains the next day's actual sales.

Using FutureSales to predict the current day's sales creates temporal leakage.

The PreviousDaySales column contains information from the previous day.

It is a valid historical feature when the prediction is made after the previous day's sales become available.

The first row has no previous-day sales, and the final row has no future-day sales. These missing values are expected because of shifting.

### Incorrect Feature Engineering → Leakage

The following feature uses future information:

FutureSales = Sales.shift(-1)

It must not be used to predict current sales because the next day's actual sales are not known at prediction time.

In [6]:
# Incorrect feature set
bad_temporal_df = sales_df.dropna().copy()

X_bad = bad_temporal_df[["FutureSales"]]

y_bad = bad_temporal_df["Sales"]

print("Incorrect Input Features:")
print(X_bad.head())

print("\nTarget:")
print(y_bad.head())

Incorrect Input Features:
   FutureSales
1        130.0
2        110.0
3        150.0
4        160.0
5        170.0

Target:
1    120
2    130
3    110
4    150
5    160
Name: Sales, dtype: int64


### Correct Feature Engineering → No Leakage

The following feature uses only historical information:

PreviousDaySales = Sales.shift(1)

It can be used to predict current sales when the previous day's sales are available.

In [7]:
# Correct feature set
good_temporal_df = sales_df.dropna(
    subset=["PreviousDaySales"]
).copy()

X_good = good_temporal_df[["PreviousDaySales"]]

y_good = good_temporal_df["Sales"]

print("Correct Input Features:")
print(X_good.head())

print("\nTarget:")
print(y_good.head())

Correct Input Features:
   PreviousDaySales
1             100.0
2             120.0
3             130.0
4             110.0
5             150.0

Target:
1    120
2    130
3    110
4    150
5    160
Name: Sales, dtype: int64


### Output Explanation

The incorrect example uses future sales information.

The correct example uses previous-day sales information.

For real forecasting projects, use chronological train-test splitting instead of randomly mixing past and future records.

Also ensure that all features are available at the exact time the prediction is supposed to be made.

## Topic 5 – Post-Outcome Features


### Step 1 – Understand the Concept

Post-Outcome Features are variables created or recorded after the event that we want to predict.

Using these features during model training can create feature leakage.

Examples:

1. Final Exam Marks used to predict whether a student will pass.
2. Claim Settlement Amount used to predict claim approval.
3. Delivery Completion Time used to predict whether an order will be delivered.
4. Hospital Discharge Date used to predict whether a patient will be admitted.
5. Loan Recovery Amount used to predict loan default.

These features may contain information about the outcome itself.

Correct approach:

Identify the prediction time and include only features that are available at that time.


### Step 2 – Demonstrate the Concept

Real-world example: Hospital Readmission Prediction

A hospital wants to predict whether a patient will be readmitted within 30 days after discharge.

Possible features available at discharge:
- Patient Age
- Previous Admissions
- Diagnosis
- Length of Current Stay

Incorrect feature:
- Actual Readmission Date

The actual readmission date is known only after the patient is readmitted.

Using it to predict readmission creates leakage.

Correct approach:

Use information available at the time of discharge.



### Step 3 – Implement the Concept

We will create a patient dataset containing an outcome-related feature called ReadmissionDate.

We will compare the incorrect and correct feature sets.

In [8]:
# Create sample hospital dataset
hospital_df = pd.DataFrame({
    "Age": [25, 45, 60, 35, 70, 55, 40, 65],
    "PreviousAdmissions": [0, 2, 4, 1, 5, 3, 1, 4],
    "LengthOfStay": [2, 5, 8, 3, 10, 6, 4, 9],
    "ReadmissionDate": [
        "2026-01-10",
        None,
        "2026-01-15",
        None,
        "2026-01-18",
        "2026-01-20",
        None,
        "2026-01-25"
    ],
    "Readmitted": [1, 0, 1, 0, 1, 1, 0, 1]
})

hospital_df["ReadmissionDate"] = pd.to_datetime(
    hospital_df["ReadmissionDate"]
)

# Incorrect feature set
X_bad = hospital_df.drop(
    columns=["Readmitted"]
)

# Correct feature set
X_good = hospital_df[
    ["Age", "PreviousAdmissions", "LengthOfStay"]
]

y = hospital_df["Readmitted"]

print("Incorrect Features:")
print(X_bad.head())

print("\nCorrect Features:")
print(X_good.head())

Incorrect Features:
   Age  PreviousAdmissions  LengthOfStay ReadmissionDate
0   25                   0             2      2026-01-10
1   45                   2             5             NaT
2   60                   4             8      2026-01-15
3   35                   1             3             NaT
4   70                   5            10      2026-01-18

Correct Features:
   Age  PreviousAdmissions  LengthOfStay
0   25                   0             2
1   45                   2             5
2   60                   4             8
3   35                   1             3
4   70                   5            10


### Output Explanation

The incorrect feature set contains ReadmissionDate.

This feature is recorded after the prediction time and may directly reveal whether readmission occurred.

The correct feature set contains only information available at discharge.

The ReadmissionDate column should not be used as an input feature for this prediction task.

## Topic 6 – Leakage Through Aggregations


### Step 1 – Understand the Concept

Leakage Through Aggregations happens when aggregated features are calculated using information that would not be available at prediction time.

Aggregation operations include:

- Sum
- Mean
- Count
- Minimum
- Maximum
- Standard Deviation

Common leakage situations:

1. Calculating customer spending using future transactions.
2. Calculating average claim amounts using records from after the prediction date.
3. Counting a customer's total purchases using the complete dataset, including future purchases.
4. Calculating group statistics using information from the test dataset.

Correct approach:

Calculate aggregations using only the records available before or at the prediction timestamp, depending on the business requirement.



### Step 2 – Demonstrate the Concept

Real-world example: Customer Churn Prediction

An online shopping company wants to predict whether a customer will stop purchasing.

The prediction date is January 31.

Incorrect feature:

Total purchases calculated using transactions from January through March.

The March transactions happen after the prediction date.

Using them to predict churn on January 31 creates temporal leakage.

Correct feature:

Calculate total purchases using transactions from January 1 through January 31 only.

This ensures that the feature represents information available at prediction time.


### Step 3 – Implement the Concept

We will create customer transaction records.

We will calculate:

1. Incorrect aggregation using all transactions.
2. Correct aggregation using only transactions available before the prediction date.

Prediction date: January 31, 2026.

In [9]:
# Create transaction dataset
transactions = pd.DataFrame({
    "CustomerID": ["C1", "C1", "C1", "C2", "C2", "C2"],
    "TransactionDate": pd.to_datetime([
        "2026-01-05",
        "2026-01-20",
        "2026-02-10",
        "2026-01-12",
        "2026-01-25",
        "2026-03-05"
    ]),
    "Amount": [1000, 2000, 5000, 1500, 2500, 6000]
})

prediction_date = pd.Timestamp("2026-01-31")

# Incorrect aggregation: uses all transactions
bad_aggregation = transactions.groupby(
    "CustomerID"
).agg(
    TotalSpending=("Amount", "sum"),
    TransactionCount=("Amount", "count")
).reset_index()

# Correct aggregation: uses only transactions on or before prediction date
available_transactions = transactions[
    transactions["TransactionDate"] <= prediction_date
]

good_aggregation = available_transactions.groupby(
    "CustomerID"
).agg(
    TotalSpending=("Amount", "sum"),
    TransactionCount=("Amount", "count")
).reset_index()

print("Incorrect Aggregation:")
print(bad_aggregation)

print("\nCorrect Aggregation:")
print(good_aggregation)

Incorrect Aggregation:
  CustomerID  TotalSpending  TransactionCount
0         C1           8000                 3
1         C2          10000                 3

Correct Aggregation:
  CustomerID  TotalSpending  TransactionCount
0         C1           3000                 2
1         C2           4000                 2


### Output Explanation

Incorrect Aggregation:

The total spending includes transactions from February and March.

These transactions were not available on January 31.

Correct Aggregation:

Only transactions on or before January 31 are included.

Therefore, the correct aggregation represents the information available at the prediction date.

The same time-based rule should be applied to customer-level, product-level, and transaction-level features.

### Incorrect Feature Engineering → Leakage

Using the complete transaction dataset:

customer_total = transactions.groupby("CustomerID")["Amount"].sum()

This includes future transactions when the prediction date is earlier than the end of the dataset.

In [10]:
# Incorrect customer-level aggregation
incorrect_total = transactions.groupby(
    "CustomerID"
)["Amount"].sum()

print(incorrect_total)

CustomerID
C1     8000
C2    10000
Name: Amount, dtype: int64


### Correct Feature Engineering → No Leakage

Filter transactions using the prediction timestamp before calculating customer-level aggregations.

Only information available at the prediction time should be included.

In [11]:
# Correct customer-level aggregation
historical_transactions = transactions[
    transactions["TransactionDate"] <= prediction_date
]

correct_total = historical_transactions.groupby(
    "CustomerID"
)["Amount"].sum()

print(correct_total)

CustomerID
C1    3000
C2    4000
Name: Amount, dtype: int64


### Output Explanation

The incorrect aggregation includes all available transactions, including future transactions.

The correct aggregation first filters the data by prediction date and then calculates total spending.

This prevents future information from entering the feature.

## Topic 7 – Leakage Through Target Encoding


### Step 1 – Understand the Concept

Target Encoding is a categorical feature encoding technique.

It replaces each category with a statistic calculated from the target variable, commonly the target mean.

Example:

A dataset contains a City column and a Purchase column.

Target Encoding calculates the average purchase rate for each city.

However, leakage can happen if target encoding is calculated using the complete dataset before splitting into training and testing data.

In that situation, the encoded training features can contain information from test labels.

Correct approach:

1. Split the dataset first.
2. Calculate category statistics using training data only.
3. Apply the learned encoding to validation and test data.
4. Handle unseen categories safely.

For high-cardinality data, cross-fitting or out-of-fold target encoding can help reduce overfitting and leakage within training data.


### Step 2 – Demonstrate the Concept

Real-world example: Online Purchase Prediction

An e-commerce company wants to predict whether a customer will purchase a product.

Categorical feature:
- City

Target:
- Purchase (0 or 1)

Suppose the complete dataset contains purchase information from Chennai customers.

Incorrect approach:

Calculate the average purchase rate for each city using all records before splitting.

The resulting encoded values may use test labels.

Correct approach:

Calculate city-level purchase rates using training records only.

Then use those training-derived values to transform the test records.


### Step 3 – Implement the Concept

We will demonstrate incorrect and correct target encoding.

Incorrect:
Calculate city target averages before splitting the dataset.

Correct:
Split the dataset first and calculate city target averages using only training records.

In [12]:
# Create sample customer purchase data
encoding_df = pd.DataFrame({
    "City": [
        "Chennai", "Chennai", "Chennai", "Chennai",
        "Mumbai", "Mumbai", "Mumbai", "Mumbai",
        "Delhi", "Delhi", "Delhi", "Delhi",
        "Chennai", "Mumbai", "Delhi", "Chennai"
    ],
    "Age": [
        22, 25, 30, 35,
        24, 28, 32, 40,
        23, 27, 33, 38,
        29, 36, 42, 45
    ],
    "Purchase": [
        1, 0, 1, 1,
        0, 1, 0, 1,
        0, 0, 1, 0,
        1, 1, 0, 1
    ]
})

# Split first
train_df, test_df = train_test_split(
    encoding_df,
    test_size=0.25,
    random_state=42,
    stratify=encoding_df["Purchase"]
)

# Incorrect: calculate target means using the full dataset
full_city_means = encoding_df.groupby(
    "City"
)["Purchase"].mean()

bad_train = train_df.copy()
bad_test = test_df.copy()

bad_train["City_Encoded"] = bad_train["City"].map(
    full_city_means
)

bad_test["City_Encoded"] = bad_test["City"].map(
    full_city_means
)

# Correct: calculate target means using training data only
train_city_means = train_df.groupby(
    "City"
)["Purchase"].mean()

global_train_mean = train_df["Purchase"].mean()

good_train = train_df.copy()
good_test = test_df.copy()

good_train["City_Encoded"] = good_train["City"].map(
    train_city_means
)

good_test["City_Encoded"] = good_test["City"].map(
    train_city_means
).fillna(global_train_mean)

print("Full Dataset City Means:")
print(full_city_means)

print("\nTraining-Only City Means:")
print(train_city_means)

print("\nCorrectly Encoded Test Data:")
print(good_test[["City", "Purchase", "City_Encoded"]])

Full Dataset City Means:
City
Chennai    0.833333
Delhi      0.200000
Mumbai     0.600000
Name: Purchase, dtype: float64

Training-Only City Means:
City
Chennai    0.800000
Delhi      0.333333
Mumbai     0.500000
Name: Purchase, dtype: float64

Correctly Encoded Test Data:
       City  Purchase  City_Encoded
8     Delhi         0      0.333333
12  Chennai         1      0.800000
14    Delhi         0      0.333333
7    Mumbai         1      0.500000


### Output Explanation

Full Dataset City Means:

These values are calculated using both training and test records.

Therefore, they may include information from test labels.

Training-Only City Means:

These values are calculated using only training records.

The correct test encoding uses the training-derived city means.

For a city not present in training data, the training target mean is used as a fallback.

Important:

This example demonstrates leakage in full-data target encoding. In practical training, using a category's own target value to encode the same training row can also cause overfitting.

Use out-of-fold target encoding or a properly cross-fitted encoder for robust training features.

### Incorrect Feature Engineering → Leakage

Calculate target encoding before splitting:

city_encoding = encoding_df.groupby("City")["Purchase"].mean()

This uses target values from the complete dataset, including test records.

In [13]:
# Incorrect target encoding
encoding_df["City_Encoded_Bad"] = encoding_df["City"].map(
    encoding_df.groupby("City")["Purchase"].mean()
)

print(encoding_df[[
    "City",
    "Purchase",
    "City_Encoded_Bad"
]].head())

      City  Purchase  City_Encoded_Bad
0  Chennai         1          0.833333
1  Chennai         0          0.833333
2  Chennai         1          0.833333
3  Chennai         1          0.833333
4   Mumbai         0          0.600000


### Correct Feature Engineering → No Leakage

Split the dataset first.

Calculate the encoding statistics using training data only.

Apply the same training-derived mapping to test data.

For production model training, use out-of-fold encoding to avoid each training row influencing its own encoded target statistic.

In [14]:
# Correct target encoding using training data
train_means = train_df.groupby(
    "City"
)["Purchase"].mean()

training_global_mean = train_df["Purchase"].mean()

train_encoded = train_df.copy()
test_encoded = test_df.copy()

train_encoded["City_Encoded"] = (
    train_encoded["City"].map(train_means)
)

test_encoded["City_Encoded"] = (
    test_encoded["City"].map(train_means)
    .fillna(training_global_mean)
)

print("Correct Training Encoding:")
print(train_encoded[["City", "Purchase", "City_Encoded"]])

print("\nCorrect Testing Encoding:")
print(test_encoded[["City", "Purchase", "City_Encoded"]])

Correct Training Encoding:
       City  Purchase  City_Encoded
3   Chennai         1      0.800000
10    Delhi         1      0.333333
1   Chennai         0      0.800000
15  Chennai         1      0.800000
13   Mumbai         1      0.500000
6    Mumbai         0      0.500000
4    Mumbai         0      0.500000
9     Delhi         0      0.333333
5    Mumbai         1      0.500000
11    Delhi         0      0.333333
2   Chennai         1      0.800000
0   Chennai         1      0.800000

Correct Testing Encoding:
       City  Purchase  City_Encoded
8     Delhi         0      0.333333
12  Chennai         1      0.800000
14    Delhi         0      0.333333
7    Mumbai         1      0.500000


### Output Explanation

The correct approach calculates the encoding statistics using only training data.

The test data is transformed using the same mapping.

This prevents test labels from influencing the training-derived encoding values.

For production use, consider cross-fitted target encoding to prevent training-row self-leakage and overfitting.

## Topic 8 – Detecting Leakage


### Step 1 – Understand the Concept

Detecting Leakage means identifying features or preprocessing steps that allow unavailable information to enter the model.

Common warning signs:

1. Extremely high model accuracy.
2. A feature has an unusually strong relationship with the target.
3. Model performance drops significantly on new data.
4. Features contain information recorded after the target outcome.
5. Training and testing datasets share information that should be separate.
6. Aggregations include future records.
7. Preprocessing was fitted using the complete dataset.

Methods to detect leakage:

- Check feature names and meanings.
- Understand when each feature becomes available.
- Check correlations between features and target.
- Compare training and testing performance.
- Review the data collection timeline.
- Verify the preprocessing and feature engineering workflow.

Important:

High accuracy or high correlation alone does not prove leakage. These are warning signs that require investigation.



### Step 2 – Demonstrate the Concept

Real-world example: Credit Card Fraud Detection

A bank wants to predict whether a transaction is fraudulent.

Available features:
- Transaction Amount
- Transaction Time
- Merchant Category
- Customer Location

Suspicious feature:
- Fraud Investigation Result

If Fraud Investigation Result is recorded after a transaction is investigated, it cannot be used to predict fraud at transaction time.

A model may achieve very high accuracy when this feature is included.

To detect the problem:

1. Check the feature definition.
2. Check when the feature is created.
3. Compare its availability with the prediction timestamp.
4. Remove the feature if it contains future information.



### Step 3 – Implement the Concept

We will create a sample fraud dataset.

We will inspect the relationship between the input features and the target.

A feature that directly reveals the target will be identified as suspicious.

In [15]:
# Create sample fraud dataset
np.random.seed(42)

n = 500

amount = np.random.randint(100, 50000, n)

transaction_hour = np.random.randint(0, 24, n)

location_risk = np.random.randint(1, 6, n)

# Create fraud target
fraud = (
    (amount > 35000) |
    ((transaction_hour < 5) & (location_risk >= 4))
).astype(int)

# Create leaked investigation result
investigation_result = fraud.copy()

fraud_df = pd.DataFrame({
    "Amount": amount,
    "TransactionHour": transaction_hour,
    "LocationRisk": location_risk,
    "InvestigationResult": investigation_result,
    "Fraud": fraud
})

# Check correlations
correlation = fraud_df.corr(numeric_only=True)

print("Feature Correlation with Target:")
print(
    correlation["Fraud"]
    .drop("Fraud")
    .sort_values(ascending=False)
)

print("\nCorrelation Matrix:")
print(correlation.round(3))

Feature Correlation with Target:
InvestigationResult    1.000000
Amount                 0.694803
LocationRisk           0.054856
TransactionHour       -0.129349
Name: Fraud, dtype: float64

Correlation Matrix:
                     Amount  TransactionHour  LocationRisk  \
Amount                1.000            0.062        -0.052   
TransactionHour       0.062            1.000         0.044   
LocationRisk         -0.052            0.044         1.000   
InvestigationResult   0.695           -0.129         0.055   
Fraud                 0.695           -0.129         0.055   

                     InvestigationResult  Fraud  
Amount                             0.695  0.695  
TransactionHour                   -0.129 -0.129  
LocationRisk                       0.055  0.055  
InvestigationResult                1.000  1.000  
Fraud                              1.000  1.000  


### Output Explanation

The InvestigationResult feature has a perfect relationship with the Fraud target in this sample dataset.

This is suspicious because the investigation result is available only after the investigation process.

The feature should not be used to predict fraud at transaction time.

The correlation analysis helps identify suspicious features, but we must also check the business meaning and availability time of each feature.

### Additional Leakage Detection – Compare Model Performance

We will compare a model using legitimate transaction features with a model using the leaked InvestigationResult feature.

In [16]:
# Define legitimate and leaked feature sets
X_correct = fraud_df[
    ["Amount", "TransactionHour", "LocationRisk"]
]

X_leaked = fraud_df[
    ["Amount", "TransactionHour", "LocationRisk",
     "InvestigationResult"]
]

y = fraud_df["Fraud"]

# Split dataset
X_train_good, X_test_good, y_train, y_test = train_test_split(
    X_correct,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_bad = X_train_good.copy()
X_test_bad = X_test_good.copy()

X_train_bad["InvestigationResult"] = (
    fraud_df.loc[X_train_good.index, "InvestigationResult"]
)

X_test_bad["InvestigationResult"] = (
    fraud_df.loc[X_test_good.index, "InvestigationResult"]
)

# Model without leakage
good_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

good_model.fit(X_train_good, y_train)

good_predictions = good_model.predict(X_test_good)

# Model with leakage
bad_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

bad_model.fit(X_train_bad, y_train)

bad_predictions = bad_model.predict(X_test_bad)

# Compare accuracy
print(
    "Accuracy Without Leakage:",
    accuracy_score(y_test, good_predictions)
)

print(
    "Accuracy With Leakage:",
    accuracy_score(y_test, bad_predictions)
)

Accuracy Without Leakage: 1.0
Accuracy With Leakage: 1.0


### Output Explanation

The model without leakage uses only transaction-time information.

The model with leakage also uses InvestigationResult, which reveals the fraud outcome.

The leaked model may achieve perfect or unusually high accuracy.

This is not a valid real-world result because InvestigationResult is not available when the transaction prediction is made.

Always investigate unusually high model performance before accepting it.

## Topic 9 – Preventing Leakage


### Step 1 – Understand the Concept

Preventing Leakage means designing the data preparation and Machine Learning workflow so that unavailable information does not enter model training or evaluation.

Important practices:

1. Split the dataset before fitting preprocessing operations.
2. Use only features available at prediction time.
3. Remove target-derived and post-outcome features.
4. Use chronological splitting for time-dependent data.
5. Calculate aggregations using only historical information.
6. Fit target encoding using training data only.
7. Use out-of-fold encoding when appropriate.
8. Use Scikit-learn Pipelines for preprocessing and model training.
9. Evaluate the model on genuinely unseen data.
10. Document the prediction timestamp and feature availability.

A Pipeline helps ensure that learned preprocessing steps are fitted correctly within each training fold.


### Step 2 – Demonstrate the Concept

Real-world example: Customer Churn Prediction

A telecom company wants to predict whether a customer will leave within the next 30 days.

Correct workflow:

1. Identify the prediction date.
2. Collect customer features available on that date.
3. Exclude information recorded after that date.
4. Split the data into training and testing sets.
5. Fit missing-value handling and scaling using training data.
6. Train the model.
7. Transform test data using the fitted preprocessing steps.
8. Evaluate performance on test data.

This workflow prevents future information and test data statistics from influencing model training.


### Step 3 – Implement the Concept

We will create a complete Scikit-learn Pipeline.

The Pipeline will:

1. Split training and testing data.
2. Handle missing values using SimpleImputer.
3. Standardize numerical features using StandardScaler.
4. Train a Logistic Regression model.
5. Evaluate performance on test data.

All preprocessing steps are fitted using training data only.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Create sample customer churn dataset
churn_df = pd.DataFrame({
    "Age": [22, 25, 30, 35, 40, 45, 50, 55, 60, 28,
            32, 38, 42, 48, 52, 58, 24, 34, 44, 54],
    "MonthlyCharges": [100, 120, 150, 180, 200, 220, 250, 280, 300, 130,
                       160, 190, 210, 230, 260, 290, 110, 170, 240, 270],
    "Tenure": [2, 3, 5, 8, 12, 15, 20, 24, 30, 4,
               7, 10, 14, 18, 22, 27, 3, 9, 16, 25],
    "Churn": [1, 1, 1, 0, 0, 0, 0, 0, 0, 1,
              1, 0, 0, 0, 0, 0, 1, 1, 0, 0]
})

# Add some missing values
churn_df.loc[2, "MonthlyCharges"] = np.nan
churn_df.loc[7, "Age"] = np.nan

# Separate features and target
X = churn_df.drop("Churn", axis=1)

y = churn_df["Churn"]

# Split data before preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Create Pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Train Pipeline
pipeline.fit(X_train, y_train)

# Predict test data
predictions = pipeline.predict(X_test)

# Evaluate model
print("Test Accuracy:",
      accuracy_score(y_test, predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

Test Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         3
           1       1.00      1.00      1.00         2

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



### Output Explanation

The Pipeline performs preprocessing and model training in the correct order.

SimpleImputer:
Fills missing values using statistics learned from training data.

StandardScaler:
Learns the scaling values from training data.

LogisticRegression:
Trains the classification model using the transformed training features.

During prediction, the test data passes through the same fitted preprocessing steps.

The test dataset is not used to calculate the training imputation or scaling statistics.

This is a safer workflow for preventing train-test leakage.

# Final Comparison – Incorrect vs Correct Feature Engineering

| Leakage Type | Incorrect Feature Engineering | Correct Feature Engineering |
|---|---|---|
| Target Leakage | Use target-derived information as an input | Use legitimate input features |
| Train-Test Leakage | Fit preprocessing on the complete dataset | Fit preprocessing on training data only |
| Temporal Leakage | Use future information | Use historical information |
| Post-Outcome Features | Use features recorded after the outcome | Use features available at prediction time |
| Aggregation Leakage | Aggregate future transactions | Aggregate historical transactions only |
| Target Encoding Leakage | Calculate target means before splitting | Calculate encoding using training data only |
| Preprocessing Leakage | Calculate statistics using test data | Use a fitted training preprocessing Pipeline |

## General Rule

Before creating any feature, ask:

1. Was this information available at prediction time?
2. Does this feature directly or indirectly reveal the target?
3. Does this feature contain information from the future?
4. Was this feature calculated using test data?
5. Was this feature calculated using information from after the outcome?

If the answer reveals unavailable information, the feature must be redesigned or removed.

# Final Summary – Feature Leakage

In this notebook, we learned:

1. What is Feature Leakage?
2. Target Leakage.
3. Train-Test Leakage.
4. Temporal Leakage.
5. Post-Outcome Features.
6. Leakage Through Aggregations.
7. Leakage Through Target Encoding.
8. Detecting Leakage.
9. Preventing Leakage.

## Important Concepts

- Feature leakage makes model performance look better than it really is.
- Target leakage occurs when input features reveal target information.
- Train-test leakage happens when test information influences training.
- Temporal leakage happens when future information is used for past predictions.
- Post-outcome features are recorded after the event being predicted.
- Aggregations must respect the prediction timestamp.
- Target encoding must not use test labels.
- Feature availability must be checked before model training.
- Scikit-learn Pipelines help prevent preprocessing leakage.

## Final Learning

A Machine Learning model should learn only from information that would genuinely be available when the prediction is made.

High accuracy is not useful if it comes from leaked information.

**Correct Feature Engineering → Reliable Model Evaluation → Better Real-World Predictions.**